# PicoRV32 UART Monitor — 宿主机串口读取 & 结果对比

在**宿主机 (PC)** 上运行。USB-TTL 接在 PC 端，读取 PYNQ 板上 PicoRV32 的 UART 输出。

## 硬件连接

```
PYNQ-Z2 PMODA              USB-TTL             PC
  Pin 1 (Y18, TX) -------> RXD                  |
  Pin 5 (GND)     -------> GND    ---USB--->  /dev/ttyUSBx (Linux)
                                               COMx (Windows)
```

## 前置条件

1. PYNQ 板已加载 `cim_rv32_soc.bit`
2. USB-TTL 接线正确，插入 PC
3. 宿主机已安装 `pyserial`: `pip install pyserial`
4. `fw_hex_batch/golden_results.txt` 已生成（来自 `prepare_picorv32_env.ipynb`）

## 1. 检测串口设备

In [7]:
import os, sys, time, glob, platform

# Detect OS and list serial ports
if platform.system() == "Windows":
    # Windows: scan COM1-COM20
    import serial.tools.list_ports
    ports = list(serial.tools.list_ports.comports())
    print("Available COM ports:")
    for p in ports:
        print(f"  {p.device:10s}  {p.description}")
    if not ports:
        print("  None found! Check Device Manager.")
else:
    # Linux / Mac
    patterns = ["/dev/ttyUSB*", "/dev/ttyACM*", "/dev/cu.usbserial*", "/dev/cu.wchusbserial*"]
    serial_ports = []
    for pat in patterns:
        serial_ports.extend(glob.glob(pat))
    print("Available serial ports:")
    if serial_ports:
        for p in sorted(serial_ports):
            print(f"  {p}")
    else:
        print("  None found!")
        print("  Linux: check dmesg | tail")
        print("  Mac:   ls /dev/cu.usb*")

Available serial ports:
  /dev/ttyUSB0
  /dev/ttyUSB1
  /dev/ttyUSB2


## 2. 配置串口参数

In [8]:
# ========== MODIFY THESE ==========
SERIAL_PORT = "/dev/ttyUSB0"   # Linux: /dev/ttyUSB0, Windows: "COM3", Mac: /dev/cu.usbserial-xxx
BAUD_RATE = 115200
TIMEOUT_SEC = 60               # Max wait time (seconds)
GOLDEN_FILE = "fw_hex_batch/golden_results.txt"  # Path to golden results
# ==================================

print(f"Serial port: {SERIAL_PORT}")
print(f"Baud rate:   {BAUD_RATE}")
print(f"Timeout:     {TIMEOUT_SEC}s")
print(f"Golden file: {GOLDEN_FILE} ({'EXISTS' if os.path.exists(GOLDEN_FILE) else 'NOT FOUND'})")

Serial port: /dev/ttyUSB0
Baud rate:   115200
Timeout:     60s
Golden file: fw_hex_batch/golden_results.txt (NOT FOUND)


## 3. 读取 UART 输出

运行此 Cell 后，在 PYNQ 板上按 **BTN0** 复位 PicoRV32，固件会重新执行并通过 UART 输出结果。

如果 PYNQ 板刚加载完 bitstream，数据可能已经发完。这种情况下必须按 BTN0 重发。

In [ ]:
import serial

print(f"Opening {SERIAL_PORT} @ {BAUD_RATE} baud...")
print(">>> Press BTN0 on PYNQ board to restart PicoRV32 <<<")
print("Waiting for UART output...\n")
print("-" * 60)

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    ser.reset_input_buffer()

    uart_output = ""
    start_time = time.time()
    done = False

    while time.time() - start_time < TIMEOUT_SEC:
        raw = ser.readline()
        if raw:
            line = raw.decode(errors="ignore").rstrip("\r\n")
            if line:
                print(line)
                uart_output += line + "\n"
            if "=== Done ===" in line:
                done = True
                break

    ser.close()
    elapsed = time.time() - start_time
    print("-" * 60)

    if done:
        print(f"\nInference complete! Elapsed: {elapsed:.1f}s")
    else:
        print(f"\nTimeout ({TIMEOUT_SEC}s)! Received {len(uart_output)} chars")
        if not uart_output:
            print("\nNo data received. Checklist:")
            print("  [1] USB-TTL RXD <-- PMODA Pin 1 (Y18)?")
            print("  [2] GND connected?")
            print("  [3] PYNQ bitstream loaded? (LED1 heartbeat blinking?)")
            print("  [4] Pressed BTN0 after this Cell started?")
            print(f"  [5] Correct port? (current: {SERIAL_PORT})")

except serial.SerialException as e:
    print(f"Serial error: {e}")
    uart_output = ""

Opening /dev/ttyUSB0 @ 115200 baud...
>>> Press BTN0 on PYNQ board to restart PicoRV32 <<<
Waiting for UART output...

------------------------------------------------------------


## 4. 解析结果

In [ ]:
import numpy as np

def parse_uart_output(text):
    result = {
        "predicted": None, "expected": None, "correct": None,
        "logits": [], "fc1_ok": False, "fc2_ok": False, "raw": text
    }
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("Predicted:"):
            try: result["predicted"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif line.startswith("Expected:"):
            try: result["expected"] = int(line.split(":")[1].strip())
            except ValueError: pass
        elif "CORRECT" in line:
            result["correct"] = True
        elif "WRONG" in line:
            result["correct"] = False
        elif line.startswith("Logits:"):
            nums = line.replace("Logits:", "").strip().split()
            result["logits"] = [int(x) for x in nums if x.lstrip("-").isdigit()]
        elif "FC1: done" in line:
            result["fc1_ok"] = True
        elif "FC2: computing" in line:
            result["fc2_ok"] = True
    return result

if uart_output:
    result = parse_uart_output(uart_output)
    print("=" * 55)
    print("  PicoRV32 FPGA Inference Result")
    print("=" * 55)
    print(f"  FC1 (784->16, ReLU): {'OK' if result['fc1_ok'] else 'N/A'}")
    print(f"  FC2 (16->10):        {'OK' if result['fc2_ok'] else 'N/A'}")
    print(f"  Predicted: {result['predicted']}")
    print(f"  Expected:  {result['expected']}")
    status = 'CORRECT' if result['correct'] else ('WRONG' if result['correct'] is not None else 'UNKNOWN')
    print(f"  Result:    {status}")
    if result["logits"]:
        print(f"  Logits:    {result['logits']}")
        print(f"  ArgMax:    {np.argmax(result['logits'])}")
    print("=" * 55)
else:
    result = None
    print("No UART output to parse")

## 5. 与 Golden Model 对比

In [ ]:
if result and result["predicted"] is not None and os.path.exists(GOLDEN_FILE):
    golden = {}
    with open(GOLDEN_FILE) as f:
        for line in f:
            if line.startswith("#"): continue
            parts = line.strip().split()
            if len(parts) >= 4:
                golden[int(parts[0])] = {
                    "label": int(parts[1]),
                    "pred": int(parts[2]),
                    "ok": int(parts[3])
                }
    print(f"Golden results loaded: {len(golden)} images")

    exp_label = result["expected"]
    hw_pred = result["predicted"]

    # Find matching image by label
    matched = [(idx, g) for idx, g in golden.items() if g["label"] == exp_label]

    if matched:
        idx, g = matched[0]
        print(f"\nMatched golden image #{idx}")
        print(f"  Golden:   label={g['label']}, pred={g['pred']}, {'correct' if g['ok'] else 'wrong'}")
        print(f"  Hardware: label={exp_label}, pred={hw_pred}")
        print()
        if hw_pred == g["pred"]:
            print("  +------------------------------------------+")
            print("  |  MATCH: Hardware == Golden (bit-exact!)   |")
            print("  +------------------------------------------+")
        else:
            print("  +------------------------------------------+")
            print("  |  MISMATCH: HW pred != Golden pred         |")
            print("  +------------------------------------------+")

        # Show all golden results for context
        print(f"\n--- Golden Model Summary ({len(golden)} images) ---")
        g_correct = sum(1 for v in golden.values() if v["ok"])
        print(f"Golden accuracy: {g_correct}/{len(golden)} ({100*g_correct/len(golden):.0f}%)")
    else:
        print(f"No golden record with label={exp_label}")

elif result and result["predicted"] is not None:
    print(f"Golden file not found: {GOLDEN_FILE}")
    print("Run prepare_picorv32_env.ipynb first to generate it")
else:
    print("No inference result to compare")

## 6. 多次测试（循环读取）

每次按 BTN0 复位后自动捕获一次输出。
因为每个 bitstream 只 bake 了一张图，所以每次结果相同——主要用于验证硬件稳定性。

In [ ]:
N_RUNS = 3  # Number of consecutive tests

print(f"Will capture {N_RUNS} runs. Press BTN0 between each run.")
print("=" * 60)

all_results = []
try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)

    for run in range(N_RUNS):
        print(f"\n--- Run {run+1}/{N_RUNS}: Press BTN0 now... ---")
        ser.reset_input_buffer()

        output = ""
        t0 = time.time()
        while time.time() - t0 < TIMEOUT_SEC:
            raw = ser.readline()
            if raw:
                line = raw.decode(errors="ignore").rstrip("\r\n")
                if line:
                    print(f"  {line}")
                    output += line + "\n"
                if "=== Done ===" in line:
                    break

        r = parse_uart_output(output)
        all_results.append(r)
        elapsed = time.time() - t0

        if r["predicted"] is not None:
            status = "CORRECT" if r["correct"] else "WRONG"
            print(f"  >> pred={r['predicted']}, expected={r['expected']}, {status} ({elapsed:.1f}s)")
        else:
            print(f"  >> No valid output ({elapsed:.1f}s)")

    ser.close()

    # Summary
    print("\n" + "=" * 60)
    print(f"{'Run':>4} {'Pred':>5} {'Expect':>7} {'Result':>8} {'Logits'}")
    print("-" * 60)
    for i, r in enumerate(all_results):
        pred = r['predicted'] if r['predicted'] is not None else '-'
        exp = r['expected'] if r['expected'] is not None else '-'
        status = 'CORRECT' if r['correct'] else ('WRONG' if r['correct'] is not None else 'N/A')
        logits = str(r['logits']) if r['logits'] else '-'
        print(f"{i+1:4d} {str(pred):>5} {str(exp):>7} {status:>8} {logits}")
    print("-" * 60)

    # Check consistency
    preds = [r['predicted'] for r in all_results if r['predicted'] is not None]
    if len(preds) > 1 and len(set(preds)) == 1:
        print(f"All {len(preds)} runs consistent: pred={preds[0]}")
    elif len(preds) > 1:
        print(f"WARNING: inconsistent predictions across runs!")

except serial.SerialException as e:
    print(f"Serial error: {e}")

## 7. 手动粘贴模式

如果串口读取不方便，也可以从其他终端软件（minicom / PuTTY / screen）复制输出，粘贴到下方进行解析。

In [ ]:
# Paste UART output here (from minicom / PuTTY / screen)
manual_output = """
=== CIM SoC RISC-V MNIST Inference ===
FC1: loading...
FC1: computing...
FC1: done
FC2: loading...
FC2: computing...

--- Result ---
Predicted: 7
Expected:  7
>>> CORRECT <<<
Logits: -16 -31 -1 18 -21 -1 -63 31 -13 6
=== Done ===
"""

# Set to True to use the pasted output
USE_MANUAL = False

if USE_MANUAL and manual_output.strip():
    result_manual = parse_uart_output(manual_output)
    print("Parsed from manual input:")
    print(f"  Predicted: {result_manual['predicted']}")
    print(f"  Expected:  {result_manual['expected']}")
    status = 'CORRECT' if result_manual['correct'] else 'WRONG'
    print(f"  Result:    {status}")
    if result_manual['logits']:
        print(f"  Logits:    {result_manual['logits']}")

    # Compare with golden
    if os.path.exists(GOLDEN_FILE):
        golden = {}
        with open(GOLDEN_FILE) as f:
            for line in f:
                if line.startswith("#"): continue
                parts = line.strip().split()
                if len(parts) >= 4:
                    golden[int(parts[0])] = {"label": int(parts[1]), "pred": int(parts[2])}
        matched = [(k,v) for k,v in golden.items() if v["label"] == result_manual["expected"]]
        if matched:
            idx, g = matched[0]
            match = result_manual["predicted"] == g["pred"]
            print(f"\n  Golden #{idx}: pred={g['pred']}")
            print(f"  {'MATCH (bit-exact!)' if match else 'MISMATCH!'}")
else:
    print("Manual mode disabled. Set USE_MANUAL = True and paste output above.")

## 总结

本 notebook 在宿主机 PC 上运行，通过 USB-TTL 读取 PYNQ 板上 PicoRV32 的 UART 输出。

**验证链路：**

```
宿主机 Python Golden (基准)
        |
        v
VCS 仿真 20张图 (功能验证)     <-- run_tb_rv32_batch.sh
        |
        v
FPGA 硬件 1张图 (上板验证)     <-- 本 notebook 读取 UART
        |
        v
PS 版 20张图 (交叉验证)       <-- picorv32_small_mlp_pynq.ipynb
```

全部 bit-exact 一致 = PicoRV32 RISC-V 可替代 ARM PS 控制 CIM 加速器